# Energy Consumption Forecasting Pipeline
### Single Meter | 1-hour intervals | Calendar + Shift + Shutdown Features

**Key Features:**
- **Working calendar**: Public holidays (Gujarat), weekends, working days/hours
- **Shift patterns**: Morning (06-14), Afternoon (14-22), Night/Off (22-06)
- **Time since last shutdown**: Hours since meter was last inactive, restart detection

**Pipeline:**
1. Data loading from MongoDB
2. Cleaning & resampling (1-hour)
3. Calendar / Shift / Shutdown pattern analysis
4. Feature engineering (calendar-driven, no weather)
5. XGBoost — short-term forecasting (1-2 days)
6. Tuned Two-Stage XGBoost (soft-blend + recency scaling + DOW profiles) — long-term future month forecast
7. Anomaly detection
8. Recursive XGBoost — next 48h future forecast

## 1. Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.ensemble import IsolationForest
import holidays

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13
print('All imports OK')

All imports OK


## 2. Data Loading

In [ ]:
from pymongo import MongoClient, DESCENDING

MONGO_URI = "string"
DB_NAME = "IOTDeviceMonitor"
COLLECTION = "FUTU00_DataMonitor"
METER_ID = "FUTU0000000004000002"
MODEL_DIR = "./models/"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION]

In [3]:
MODEL_DIR = "E:\Projects\IOT-AI-Implementation\models\MODEL02"

In [4]:
query = {"EnergymeterId": METER_ID}
cursor = collection.find(query).sort("_id", DESCENDING)
raw = list(cursor)

df_raw = pd.DataFrame(raw)
print(f"Raw shape: {df_raw.shape}")
df_raw.head(10)

Raw shape: (980984, 6)


,_id,UID,RTC,A1,EnergymeterId,timestamp
0,69d8c5a49f2e531fa0d682eb,02,2026-04-10 08:08:00,345252.6400,FUTU0000000004000002,10-04-2026 15:10:52
1,69d8c56c9f2e531fa0d67d37,02,2026-04-10 08:07:00,345252.5600,FUTU0000000004000002,10-04-2026 15:09:56
2,69d8c52c9f2e531fa0d676f5,02,2026-04-10 08:06:00,345252.5200,FUTU0000000004000002,10-04-2026 15:08:52
3,69d8c4f09f2e531fa0d6736d,02,2026-04-10 08:05:00,345252.4800,FUTU0000000004000002,10-04-2026 15:07:52
4,69d8c4b49f2e531fa0d66fc3,02,2026-04-10 08:04:00,345252.4800,FUTU0000000004000002,10-04-2026 15:06:52
5,69d8c47b9f2e531fa0d66c81,02,2026-04-10 08:03:00,345251.1600,FUTU0000000004000002,10-04-2026 15:05:55
6,69d8c43d9f2e531fa0d668c5,02,2026-04-10 08:02:00,345249.9200,FUTU0000000004000002,10-04-2026 15:04:52
7,69d8c4009f2e531fa0d66583,02,2026-04-10 08:01:00,345249.7600,FUTU0000000004000002,10-04-2026 15:03:52
8,69d8c3e29f2e531fa0d663ad,02,2026-04-10 08:00:00,345249.7200,FUTU0000000004000002,10-04-2026 15:03:22
9,69d8c3db9f2e531fa0d66359,02,2026-04-10 08:00:00,345249.7200,FUTU0000000004000002,10-04-2026 15:03:15


In [56]:
df_raw.tail(10)

,_id,UID,RTC,A1,EnergymeterId,timestamp
980974,670f5a50f27e74074710d499,02,2024-10-16 04:46:00,79908.7500,FUTU0000000004000002,16-10-2024 11:46:48
980975,670f5a21f27e74074710d468,02,2024-10-16 04:45:00,79907.5200,FUTU0000000004000002,16-10-2024 11:46:01
980976,670f5a20f27e74074710d463,02,2024-10-16 04:45:00,79907.5200,FUTU0000000004000002,16-10-2024 11:46:00
980977,670f59cef27e74074710d42a,02,2024-10-16 04:44:00,79906.7100,FUTU0000000004000002,16-10-2024 11:44:38
980978,670f5994f27e74074710d3f2,02,2024-10-16 04:43:00,79906.1100,FUTU0000000004000002,16-10-2024 11:43:40
980979,670f5960f27e74074710d3c0,02,2024-10-16 04:42:00,79905.7500,FUTU0000000004000002,16-10-2024 11:42:48
980980,670f5939f27e74074710d39a,02,2024-10-16 04:41:00,79905.7500,FUTU0000000004000002,16-10-2024 11:42:09
980981,670f5939f27e74074710d395,02,2024-10-16 04:40:00,79905.3100,FUTU0000000004000002,16-10-2024 11:42:09
980982,670f58f4f27e74074710d35c,02,2024-10-16 04:40:00,79905.3100,FUTU0000000004000002,16-10-2024 11:41:00
980983,670f58f4f27e74074710d35b,02,2024-10-16 04:40:00,79905.3100,FUTU0000000004000002,16-10-2024 11:41:00


In [55]:
df_temp = df_raw.copy()
df_temp['A1_num'] = pd.to_numeric(df_temp['A1'], errors='coerce')

zero_rows = df_temp[df_temp['A1_num'] == 0]

print(zero_rows)
print("Count:", len(zero_rows))  

                             _id UID                 RTC      A1  \
72352   69a501d2c52ba25d1bcb8fd4  02 2026-03-02 01:45:00  0.0000   
72355   69a501d2c52ba25d1bcb8faa  02 2026-03-02 01:45:00  0.0000   
72358   69a501c6c52ba25d1bcb8f78  02 2026-03-02 01:45:00  0.0000   
94937   699afe8e9d0d02e4bcbd2be8  02 2026-02-20 05:28:00  0.0000   
94942   699afe8a9d0d02e4bcbd2af2  02 2026-02-20 05:28:00  0.0000   
...                          ...  ..                 ...     ...   
788313  67ac2a47752c438f5c325dbb  02 2025-02-11 20:49:00  0.0000   
788582  67ac2806752c438f5c3254b1  02 2025-02-11 16:30:00  0.0000   
788657  67ac2739752c438f5c3251fd  02 2025-02-11 15:28:00  0.0000   
788922  67ac2509752c438f5c324951  02 2025-02-11 11:16:00  0.0000   
811491  6799cec8f2309cf3992565d4  02 2025-01-29 05:06:00  0.0000   

               EnergymeterId            timestamp  A1_num  
72352   FUTU0000000004000002  02-03-2026 08:49:46     0.0  
72355   FUTU0000000004000002  02-03-2026 08:49:46     0.0  
723

## 3. Cleaning & Resampling

In [57]:
df = df_raw.copy()
print(f"S1: {len(df)}")

df = df.rename(columns={
    'A1': 'total_kwh',
    'RTC': 'rtc_timestamp',
    'EnergymeterId': 'meter_id'
})
print(f"S2: {len(df)}")


df = df.sort_values(['meter_id', 'rtc_timestamp'])

df['total_kwh'] = pd.to_numeric(df['total_kwh'], errors='coerce')
df['rtc_timestamp'] = pd.to_datetime(df['rtc_timestamp'], errors='coerce')
df = df.dropna(subset=['rtc_timestamp', 'total_kwh'])
print(f"S3: {len(df)}")


df.loc[df['total_kwh'] == 0, 'total_kwh'] = np.nan
df = df.dropna()

print(f"S4: {len(df)}")

df = (
    df.set_index('rtc_timestamp')
      .resample('1h')['total_kwh']
      .mean()
      .reset_index()
)
print(f"S5: {len(df)}")

# Hourly consumption = difference in cumulative meter reading
df['consumption_kwh'] = df['total_kwh'].diff().clip(lower=0)
df.loc[df['consumption_kwh'].isna(), 'consumption_kwh'] = 0

print(f"After cleaning: {df.shape}")
print(f"Date range: {df['rtc_timestamp'].min()} → {df['rtc_timestamp'].max()}")

# --- Working calendar (Gujarat public holidays) ---
_gj_holidays = holidays.India(state='GJ', years=range(2023, 2027))

df['is_holiday'] = df['rtc_timestamp'].dt.date.apply(
    lambda d: 1 if d in _gj_holidays else 0
)
df['is_weekend'] = (df['rtc_timestamp'].dt.dayofweek >= 5).astype(int)
df['is_working_day'] = ((df['is_weekend'] == 0) & (df['is_holiday'] == 0)).astype(int)
df['is_working_hour'] = (
    (df['rtc_timestamp'].dt.hour >= 9) &
    (df['rtc_timestamp'].dt.hour <= 18) &
    (df['is_working_day'] == 1)
).astype(int)

# --- Shift pattern ---
def _get_shift(hour):
    if 6  <= hour < 14: return 1   # Morning shift
    if 14 <= hour < 22: return 2   # Afternoon shift
    return 0                       # Night / off

df['shift'] = df['rtc_timestamp'].dt.hour.apply(_get_shift)

# --- Time since last shutdown ---
df['is_zero'] = (df['consumption_kwh'] < 0.5).astype(int)
_shutdown_grp = (df['is_zero'] != df['is_zero'].shift()).cumsum()
df['hours_since_shutdown'] = df.groupby(_shutdown_grp).cumcount()
df.loc[df['is_zero'] == 1, 'hours_since_shutdown'] = 0
df['hours_since_shutdown'] = df['hours_since_shutdown'].clip(upper=168)

print(f"\nHoliday hours in dataset: {df['is_holiday'].sum()}")
print(f"Working hours : {df['is_working_hour'].sum()}")
print(f"Shift distribution:\n{df['shift'].value_counts().sort_index().to_string()}")
print(f"\nConsumption stats:\n{df['consumption_kwh'].describe().to_string()}")


S1: 980984
S2: 980984
S3: 980984
S4: 980917
S5: 12989
After cleaning: (12989, 3)
Date range: 2024-10-16 04:00:00 → 2026-04-10 08:00:00

Holiday hours in dataset: 672
Working hours : 3650
Shift distribution:
shift
0    4330
1    4331
2    4328

Consumption stats:
count    12989.000000
mean        19.570873
std         12.161099
min          0.000000
25%         11.518388
50%         21.993756
75%         27.919964
max         63.685547


In [ ]:
# Fetch & merge
# Data summary after cleaning
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 3.1 Calendar, Shift & Shutdown Pattern Analysis

> Analysing how **working calendar**, **shift patterns**, and **time since last shutdown** relate to energy consumption patterns.

In [ ]:
# 1. Working Day vs Non-Working Day consumption
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Box plot: working day vs not
labels_wd = {0: 'Non-Working', 1: 'Working'}
df['wd_label'] = df['is_working_day'].map(labels_wd)
df.boxplot(column='consumption_kwh', by='wd_label', ax=axes[0],
           patch_artist=True, grid=False,
           boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[0].set_title('Consumption: Working vs Non-Working Day')
axes[0].set_xlabel('')
axes[0].set_ylabel('kWh / hour')
fig.suptitle('')

# Box plot: holiday vs not
labels_h = {0: 'Regular Day', 1: 'Holiday'}
df['hol_label'] = df['is_holiday'].map(labels_h)
df.boxplot(column='consumption_kwh', by='hol_label', ax=axes[1],
           patch_artist=True, grid=False,
           boxprops=dict(facecolor='coral', alpha=0.5))
axes[1].set_title('Consumption: Holiday vs Regular Day')
axes[1].set_xlabel('')
axes[1].set_ylabel('kWh / hour')
fig.suptitle('')

# Weekend vs Weekday
labels_we = {0: 'Weekday', 1: 'Weekend'}
df['we_label'] = df['is_weekend'].map(labels_we)
df.boxplot(column='consumption_kwh', by='we_label', ax=axes[2],
           patch_artist=True, grid=False,
           boxprops=dict(facecolor='mediumpurple', alpha=0.5))
axes[2].set_title('Consumption: Weekday vs Weekend')
axes[2].set_xlabel('')
axes[2].set_ylabel('kWh / hour')
fig.suptitle('')
plt.tight_layout()
plt.show()


# 2. Shift-wise energy consumption
shift_labels = {0: 'Night/Off\n(22-06)', 1: 'Morning\n(06-14)', 2: 'Afternoon\n(14-22)'}
df['shift_label'] = df['shift'].map(shift_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shift_means = df.groupby('shift')['consumption_kwh'].mean()
colors_shift = ['#2c3e50', '#e67e22', '#2980b9']
axes[0].bar([shift_labels[i] for i in shift_means.index], shift_means.values,
            color=colors_shift, alpha=0.8, edgecolor='white')
axes[0].set_title('Average Hourly Consumption by Shift')
axes[0].set_ylabel('Avg kWh / hour')

df.boxplot(column='consumption_kwh', by='shift_label', ax=axes[1],
           patch_artist=True, grid=False,
           boxprops=dict(facecolor='#3498db', alpha=0.4))
axes[1].set_title('Consumption Distribution by Shift')
axes[1].set_xlabel('')
axes[1].set_ylabel('kWh / hour')
fig.suptitle('')
plt.tight_layout()
plt.show()


# 3. Hourly consumption heatmap by day of week
pivot = df.pivot_table(
    values='consumption_kwh',
    index=df['rtc_timestamp'].dt.dayofweek,
    columns=df['rtc_timestamp'].dt.hour,
    aggfunc='mean'
)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot.index = day_names

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Avg kWh/h'})
ax.set_title('Avg Hourly Consumption — Day of Week × Hour of Day')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


# 4. Time since last shutdown vs consumption
active_df = df[df['hours_since_shutdown'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(active_df['hours_since_shutdown'], active_df['consumption_kwh'],
                alpha=0.08, s=4, color='teal')
axes[0].set_xlabel('Hours Since Last Shutdown')
axes[0].set_ylabel('kWh / hour')
axes[0].set_title('Consumption vs Hours Since Shutdown')

bins = [0, 2, 6, 12, 24, 48, 168]
bin_labels = ['0-2h', '2-6h', '6-12h', '12-24h', '24-48h', '48h+']
active_df['shutdown_bin'] = pd.cut(active_df['hours_since_shutdown'], bins=bins, labels=bin_labels)
active_df.boxplot(column='consumption_kwh', by='shutdown_bin', ax=axes[1],
                  patch_artist=True, grid=False,
                  boxprops=dict(facecolor='teal', alpha=0.4))
axes[1].set_title('Consumption by Time-Since-Shutdown Bucket')
axes[1].set_xlabel('')
axes[1].set_ylabel('kWh / hour')
fig.suptitle('')
plt.tight_layout()
plt.show()


# 5. Correlation heatmap (calendar features vs consumption)
corr_cols = ['consumption_kwh', 'is_holiday', 'is_weekend', 'is_working_day',
             'is_working_hour', 'shift', 'hours_since_shutdown']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation: Calendar / Shift / Shutdown Features vs Consumption')
plt.tight_layout()
plt.show()


# 6. Monthly consumption trend
monthly = df.set_index('rtc_timestamp')['consumption_kwh'].resample('ME').sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly.index, monthly.values, width=20, color='steelblue', alpha=0.7)
ax.set_title('Total Monthly Energy Consumption (kWh)')
ax.set_ylabel('Total kWh')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()

# Clean up temp cols
df.drop(columns=['wd_label', 'hol_label', 'we_label', 'shift_label'], inplace=True, errors='ignore')

In [ ]:
# Plot raw cleaned data
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['rtc_timestamp'], df['consumption_kwh'], linewidth=0.5, color='steelblue', alpha=0.8)
axes[0].set_title('Hourly Energy Consumption — Full History')
axes[0].set_ylabel('kWh / hour')

axes[1].plot(df['rtc_timestamp'], df['total_kwh'], linewidth=0.8, color='darkorange')
axes[1].set_title('Cumulative Meter Reading')
axes[1].set_ylabel('Total kWh')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Basic stats
print(df['total_kwh'].describe())

# Missing % after resample
full_range = pd.date_range(df['rtc_timestamp'].min(), df['rtc_timestamp'].max(), freq='1h')
missing_pct = (1 - len(df) / len(full_range)) * 100
print(f"\nMissing 1h slots: {missing_pct:.2f}%")

## 4. Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Time features
    out['hour_of_day'] = out['rtc_timestamp'].dt.hour
    out['day_of_week'] = out['rtc_timestamp'].dt.dayofweek
    out['month']       = out['rtc_timestamp'].dt.month
    out['year']        = out['rtc_timestamp'].dt.year
    out['is_weekend']  = (out['day_of_week'] >= 5).astype(int)

    # Cyclical encoding
    out['hour_sin'] = np.sin(2 * np.pi * out['hour_of_day'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour_of_day'] / 24)
    out['dow_sin']  = np.sin(2 * np.pi * out['day_of_week'] / 7)
    out['dow_cos']  = np.cos(2 * np.pi * out['day_of_week'] / 7)
    out['mon_sin']  = np.sin(2 * np.pi * out['month'] / 12)
    out['mon_cos']  = np.cos(2 * np.pi * out['month'] / 12)

    # Lag features on kWh
    for lag in [1, 2, 3, 6, 12, 24]:
        out[f'kwh_lag_{lag}'] = out['total_kwh'].shift(lag)

    out['target_diff'] = out['total_kwh'] - out['kwh_lag_1']
    out = out[out['target_diff'].abs() > 1]

    # Rolling stats on kWh
    out['rolling_max_6']   = out['total_kwh'].rolling(6).max()
    out['rolling_min_6']   = out['total_kwh'].rolling(6).min()
    out['rolling_range_6'] = out['rolling_max_6'] - out['rolling_min_6']
    for w in [3, 6, 24]:
        out[f'kwh_roll_std_{w}'] = out['total_kwh'].rolling(w, min_periods=2).std()
    out['diff_2'] = out['total_kwh'].diff(2)

    # Consumption lags
    for lag in [1, 2, 3, 6, 24]:
        out[f'cons_lag_{lag}'] = out['consumption_kwh'].shift(lag)
    out['cons_roll_mean_6']  = out['consumption_kwh'].rolling(6, min_periods=1).mean()
    out['cons_roll_mean_24'] = out['consumption_kwh'].rolling(24, min_periods=1).mean()

    # --- Working calendar ---
    _gj_holidays = holidays.India(state='GJ', years=range(2023, 2027))
    out['is_holiday'] = out['rtc_timestamp'].dt.date.apply(
        lambda d: 1 if d in _gj_holidays else 0)
    out['is_working_day'] = (
        (out['rtc_timestamp'].dt.dayofweek < 5) & (out['is_holiday'] == 0)
    ).astype(int)
    out['is_working_hour'] = (
        (out['rtc_timestamp'].dt.hour >= 9) &
        (out['rtc_timestamp'].dt.hour <= 18) &
        (out['is_working_day'] == 1)
    ).astype(int)

    holiday_dates = set(_gj_holidays.keys())
    def days_to_next_holiday(d):
        for i in range(1, 8):
            if (d + pd.Timedelta(days=i)).date() in holiday_dates:
                return i
        return 7
    date_to_dnth = {d: days_to_next_holiday(pd.Timestamp(d))
                    for d in out['rtc_timestamp'].dt.normalize().unique()}
    out['days_to_next_holiday'] = out['rtc_timestamp'].dt.normalize().map(date_to_dnth)

    # --- Shift pattern ---
    out['shift'] = out['hour_of_day'].apply(
        lambda h: 1 if 6 <= h < 14 else (2 if 14 <= h < 22 else 0)
    )
    out['shift_sin'] = np.sin(2 * np.pi * out['shift'] / 3)
    out['shift_cos'] = np.cos(2 * np.pi * out['shift'] / 3)

    # --- Time since last shutdown ---
    out['is_zero'] = (out['consumption_kwh'] < 0.5).astype(int)
    _grp = (out['is_zero'] != out['is_zero'].shift()).cumsum()
    out['hours_since_shutdown'] = out.groupby(_grp).cumcount()
    out.loc[out['is_zero'] == 1, 'hours_since_shutdown'] = 0
    out['hours_since_shutdown'] = out['hours_since_shutdown'].clip(upper=168)
    out['just_restarted'] = (
        (out['hours_since_shutdown'] > 0) & (out['hours_since_shutdown'] <= 2)
    ).astype(int)

    return out


feat_df = build_features(df)
feat_df = feat_df.dropna().reset_index(drop=True)

print(f"Feature df shape: {feat_df.shape}")
print(f"Features: {list(feat_df.columns)}")
feat_df.head()

In [ ]:
# Hourly, weekly & shift patterns
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

feat_df.groupby('hour_of_day')['consumption_kwh'].mean().plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Avg Consumption by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg kWh/h')

days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
feat_df.groupby('day_of_week')['consumption_kwh'].mean().plot(ax=axes[1], marker='o', color='coral')
axes[1].set_title('Avg Consumption by Day of Week')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(days)

# Working hour vs off-hour
wh_means = feat_df.groupby('is_working_hour')['consumption_kwh'].mean()
axes[2].bar(['Off-Hours', 'Working Hours'], wh_means.values,
            color=['#95a5a6', '#27ae60'], alpha=0.8)
axes[2].set_title('Avg Consumption: Working vs Off-Hours')
axes[2].set_ylabel('Avg kWh/h')

plt.tight_layout()
plt.show()

In [ ]:
feat_df.columns

## 5. Train / Test Split

In [ ]:
TARGET = 'consumption_kwh'

EXCLUDED = {
    'rtc_timestamp', 'total_kwh', 'consumption_kwh', 'is_zero',
    'hour_of_day', 'day_of_week', 'month',  # cyclical sin/cos used instead
    'year', 'is_weekend',                    # redundant with dow_sin/cos
    'target_diff',
}

model_features = sorted([c for c in feat_df.columns if c not in EXCLUDED])

split_idx = int(len(feat_df) * 0.8)
train_df = feat_df.iloc[:split_idx].copy()
test_df = feat_df.iloc[split_idx:].copy()

X_train = train_df[model_features]
y_train = train_df[TARGET]
X_test = test_df[model_features]
y_test = test_df[TARGET]

print(f"Train: {train_df['rtc_timestamp'].min()} → {train_df['rtc_timestamp'].max()} | {len(train_df):,} rows")
print(f"Test:  {test_df['rtc_timestamp'].min()} → {test_df['rtc_timestamp'].max()} | {len(test_df):,} rows")
print(f"Features ({len(model_features)}): {model_features}")

In [ ]:
X_train.nunique().sort_values()

In [ ]:
y_train.describe()

## 6. XGBoost Model — Short-term Forecasting

In [ ]:
# explicitly define which columns go into the model
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_features

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features)
    ],
    remainder='drop' # Drops unused columns
)

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=800,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
    ))
])

xgb_model.fit(X_train, y_train)
print('XGBoost training complete')

In [ ]:
# Predictions
y_pred_vals = xgb_model.predict(X_test)
y_pred = pd.Series(y_pred_vals, index=X_test.index, name='predicted_kwh')

print(f"Predictions shape: {y_pred.shape}")
y_pred.head()

In [ ]:
# Metrics
xgb_mae  = mean_absolute_error(y_test, y_pred)
xgb_rmse = root_mean_squared_error(y_test, y_pred)
safe_actual = np.clip(y_test.values, 1e-6, None)
xgb_mape = np.mean(np.abs((y_test.values - y_pred.values) / safe_actual)) * 100

metrics_df = pd.DataFrame([{
    'Model': 'XGBoost',
    'MAE': round(xgb_mae, 4),
    'RMSE': round(xgb_rmse, 4),
    'MAPE%': round(xgb_mape, 2),
}])
print(f"XGBoost  MAE: {xgb_mae:.4f}  |  RMSE: {xgb_rmse:.4f}  |  MAPE: {xgb_mape:.2f}%")

## 7. Plots — Training, Testing, Residuals

In [ ]:
# Reconstruct predictions for plotting
y_train_pred = xgb_model.predict(X_train)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_df['rtc_timestamp'], y_train.values,
        label='Train (actual)', color='steelblue', linewidth=0.5, alpha=0.7)
ax.plot(test_df['rtc_timestamp'], y_test.values,
        label='Test (actual)', color='orange', linewidth=0.8)
ax.plot(test_df['rtc_timestamp'], y_pred.values,
        label='Test (predicted)', color='red', linewidth=0.8, linestyle='--')
ax.axvline(test_df['rtc_timestamp'].iloc[0], color='black', linestyle=':', linewidth=1.5, label='Train/Test split')
ax.set_title('XGBoost — Train / Test Forecast (Consumption kWh/h)')
ax.set_ylabel('kWh / hour')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into last 7 days of test
last7 = test_df.tail(24 * 7)
pred_last7 = y_pred.iloc[-len(last7):]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(last7['rtc_timestamp'], last7['consumption_kwh'].values,
        label='Actual', color='steelblue', linewidth=1.2)
ax.plot(last7['rtc_timestamp'], pred_last7.values,
        label='Predicted', color='red', linewidth=1.1, linestyle='--')
ax.set_title('XGBoost — Last 7 Days of Test Set (Zoomed)')
ax.set_ylabel('kWh / hour')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
booster = xgb_model.named_steps['model']
feat_imp = pd.Series(booster.feature_importances_, index=model_features)
top20 = feat_imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Feature Importances (XGBoost)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Anomaly Detection

In [ ]:
results = test_df[['rtc_timestamp', 'consumption_kwh']].copy()
results['predicted'] = y_pred.values
results['residual'] = results['consumption_kwh'] - results['predicted']
resid_std = results['residual'].std()
results['z_score'] = results['residual'] / resid_std

# Isolation Forest on residuals
iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
results['iso_flag']  = iso.fit_predict(results[['residual']])
results['iso_score'] = iso.decision_function(results[['residual']])

# Rule-based: |z| > 3
results['rule_flag'] = (results['z_score'].abs() > 3).astype(int)
results['anomaly']   = ((results['iso_flag'] == -1) | (results['rule_flag'] == 1)).astype(int)
results['severity']  = results['z_score'].abs().round(3)

anomalies = results[results['anomaly'] == 1].sort_values('severity', ascending=False)
print(f"Anomalies detected: {len(anomalies)} / {len(results)} ({100*len(anomalies)/len(results):.1f}%)")
anomalies.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results['rtc_timestamp'], results['consumption_kwh'], label='Actual', linewidth=0.9, color='steelblue')
ax.plot(results['rtc_timestamp'], results['predicted'], label='Predicted', linewidth=0.9, color='orange', linestyle='--')
ax.scatter(anomalies['rtc_timestamp'], anomalies['consumption_kwh'], color='red', s=25, zorder=5, label='Anomaly')
ax.set_title('Anomaly Detection on Test Set')
ax.set_ylabel('kWh / hour')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Save XGBoost Model

In [ ]:
import os
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(xgb_model, f"{MODEL_DIR}/xgb_energy_model.pkl")
joblib.dump(model_features, f"{MODEL_DIR}/model_features.pkl")

print('XGBoost model saved.')

## 10. Long-term Forecasting — Two-Stage XGBoost + Hourly Profile Decomposition

> **Strategy**: Two-stage approach to handle **bimodal** consumption (active vs shutdown days):  
> 1. **Stage 1 — Classifier**: XGBoost predicts P(active) — used as a **soft blend weight** (not hard 0/1)  
> 2. **Stage 2 — Regressor**: XGBoost (trained only on active days) predicts consumption magnitude  
> 3. **Recency scaling**: Adjusts for recent consumption trend vs. full training history  
> 4. **Decomposition**: Daily totals → hourly values via **day-of-week** profiles (7 profiles, with day-type fallback)

In [ ]:
from xgboost import XGBRegressor as XGBDaily, XGBClassifier

DAILY_HORIZON = 31
SHUTDOWN_THRESH = 50  # kWh — below this a day is considered "shutdown"

_gj_hols_nf = holidays.India(state='GJ', years=range(2023, 2028))

# Step 1: Build daily totals
hourly_df = df[['rtc_timestamp', 'consumption_kwh']].copy().dropna()

daily_df = (
    hourly_df.set_index('rtc_timestamp')['consumption_kwh']
    .resample('D').sum()
    .reset_index()
    .rename(columns={'rtc_timestamp': 'ds', 'consumption_kwh': 'y'})
)

# Calendar features (deterministic — known for future dates)
daily_df['is_holiday']    = daily_df['ds'].dt.date.apply(lambda d: 1 if d in _gj_hols_nf else 0)
daily_df['is_weekend']    = (daily_df['ds'].dt.dayofweek >= 5).astype(int)
daily_df['is_working_day'] = ((daily_df['is_weekend'] == 0) & (daily_df['is_holiday'] == 0)).astype(int)
daily_df['dow']           = daily_df['ds'].dt.dayofweek
daily_df['dow_sin']       = np.sin(2 * np.pi * daily_df['dow'] / 7)
daily_df['dow_cos']       = np.cos(2 * np.pi * daily_df['dow'] / 7)
daily_df['month']         = daily_df['ds'].dt.month
daily_df['mon_sin']       = np.sin(2 * np.pi * daily_df['month'] / 12)
daily_df['mon_cos']       = np.cos(2 * np.pi * daily_df['month'] / 12)
daily_df['day_of_month']  = daily_df['ds'].dt.day

# Binary label: 1 = active day, 0 = shutdown day
daily_df['is_active'] = (daily_df['y'] >= SHUTDOWN_THRESH).astype(int)

DAILY_FEATURES = [
    'is_holiday', 'is_weekend', 'is_working_day',
    'dow', 'dow_sin', 'dow_cos', 'month', 'mon_sin', 'mon_cos', 'day_of_month',
]

active_pct = daily_df['is_active'].mean() * 100
print(f"Daily df shape: {daily_df.shape}")
print(f"Date range: {daily_df['ds'].min().date()} → {daily_df['ds'].max().date()}")
print(f"Active days: {daily_df['is_active'].sum()}/{len(daily_df)} ({active_pct:.1f}%)")
print(f"Shutdown days: {(~daily_df['is_active'].astype(bool)).sum()}/{len(daily_df)} ({100-active_pct:.1f}%)")
print(f"Features ({len(DAILY_FEATURES)}): {DAILY_FEATURES}")
daily_df.tail(5)

In [ ]:
# Step 2: Build hourly profiles — TWO levels of granularity
# Level 1: day-of-week profiles (7 profiles) — primary, captures Mon vs Fri differences
# Level 2: day-type profiles (3 profiles) — fallback for holidays / sparse DOWs

def classify_day(ts):
    if ts.date() in _gj_hols_nf:
        return 'holiday'
    if ts.dayofweek >= 5:
        return 'weekend'
    return 'working'

hourly_df['day_type'] = hourly_df['rtc_timestamp'].apply(classify_day)
hourly_df['hour'] = hourly_df['rtc_timestamp'].dt.hour
hourly_df['dow'] = hourly_df['rtc_timestamp'].dt.dayofweek

# Only use active days for profiles (shutdown days are flat ~0 and distort averages)
active_day_dates = set(
    hourly_df.groupby(hourly_df['rtc_timestamp'].dt.date)['consumption_kwh']
    .sum().loc[lambda s: s >= SHUTDOWN_THRESH].index
)
active_mask = hourly_df['rtc_timestamp'].dt.date.isin(active_day_dates)
hourly_active = hourly_df[active_mask].copy()

day_totals = hourly_active.groupby(hourly_active['rtc_timestamp'].dt.date)['consumption_kwh'].transform('sum')
hourly_active['hour_fraction'] = hourly_active['consumption_kwh'] / day_totals.clip(lower=1e-6)

# ── Level 1: Day-of-week profiles (7 columns: 0=Mon .. 6=Sun) ──
dow_profiles = (
    hourly_active.groupby(['dow', 'hour'])['hour_fraction']
    .mean()
    .reset_index()
    .pivot(index='hour', columns='dow', values='hour_fraction')
    .fillna(0)
)
dow_profiles = dow_profiles / dow_profiles.sum()  # normalize each DOW to sum=1

# ── Level 2: Day-type profiles (fallback) ──
hourly_profiles = (
    hourly_active.groupby(['day_type', 'hour'])['hour_fraction']
    .mean()
    .reset_index()
    .pivot(index='hour', columns='day_type', values='hour_fraction')
    .fillna(0)
)
hourly_profiles = hourly_profiles / hourly_profiles.sum()

MIN_DAYS_FOR_DOW = 4  # use DOW profile only if we have enough samples
dow_day_counts = hourly_active.groupby('dow')['rtc_timestamp'].apply(
    lambda s: s.dt.date.nunique()
)
print("DOW profile sample counts:")
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for d in range(7):
    cnt = dow_day_counts.get(d, 0)
    status = "✓ DOW" if cnt >= MIN_DAYS_FOR_DOW else "→ fallback to day-type"
    print(f"  {dow_names[d]}: {cnt} active days  {status}")

print("\nDay-type hourly profiles (fraction of daily total):")
print(hourly_profiles.round(4))

# Plot both profile levels
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for col in hourly_profiles.columns:
    axes[0].plot(hourly_profiles.index, hourly_profiles[col] * 100, marker='o', ms=4, label=col)
axes[0].set_title('Day-Type Profiles (fallback)')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('% of Daily')
axes[0].set_xticks(range(24)); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for d in range(7):
    if d in dow_profiles.columns:
        axes[1].plot(dow_profiles.index, dow_profiles[d] * 100, marker='o', ms=3, label=dow_names[d])
axes[1].set_title('Day-of-Week Profiles (primary)')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('% of Daily')
axes[1].set_xticks(range(24)); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Clean temp cols
hourly_df.drop(columns=['day_type', 'hour', 'dow'], inplace=True)

In [ ]:
# Step 3: Train/val split + Two-stage model
daily_train = daily_df.iloc[:-DAILY_HORIZON].copy()
daily_val   = daily_df.iloc[-DAILY_HORIZON:].copy()

train_active = daily_train[daily_train['is_active'] == 1]
train_shutdown = daily_train[daily_train['is_active'] == 0]

print(f"Daily Train: {daily_train['ds'].min().date()} → {daily_train['ds'].max().date()} | {len(daily_train)} days")
print(f"  Active: {len(train_active)} | Shutdown: {len(train_shutdown)}")
print(f"Daily Val  : {daily_val['ds'].min().date()} → {daily_val['ds'].max().date()} | {len(daily_val)} days")

# ── Stage 1: Classifier (active vs shutdown) ──
xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(train_shutdown) / max(len(train_active), 1),
    random_state=42,
    eval_metric='logloss',
)
xgb_clf.fit(daily_train[DAILY_FEATURES], daily_train['is_active'], verbose=False)

train_clf_acc = (xgb_clf.predict(daily_train[DAILY_FEATURES]) == daily_train['is_active']).mean()
print(f"\nStage 1 — Classifier trained (active vs shutdown)")
print(f"  Train accuracy: {train_clf_acc:.1%}")

# ── Stage 2: Regressor (trained ONLY on active days) ──
xgb_reg = XGBDaily(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
)
xgb_reg.fit(train_active[DAILY_FEATURES], train_active['y'], verbose=False)

train_reg_mae = np.mean(np.abs(train_active['y'] - xgb_reg.predict(train_active[DAILY_FEATURES])))
print(f"\nStage 2 — Regressor trained (active days only: {len(train_active)} samples)")
print(f"  Train MAE: {train_reg_mae:.1f} kWh/day")
print(f"  Active-day mean: {train_active['y'].mean():.1f}, median: {train_active['y'].median():.1f} kWh/day")

In [ ]:
# Step 4: Tuned two-stage validation prediction
# ── Tuning A: Soft probability blending instead of hard 0/1 cutoff ──
# Instead of zeroing out days below the default 0.5 threshold,
# multiply regressor output by P(active). Days with P=0.36 get 36% of predicted kWh.

val_active_prob = xgb_clf.predict_proba(daily_val[DAILY_FEATURES])[:, 1]
val_consumption = np.clip(xgb_reg.predict(daily_val[DAILY_FEATURES]), 0, None)

# Soft blending: pred = P(active) * regressor_output
# Apply a floor: if P < 0.05 treat as true shutdown (avoid noise)
PROB_FLOOR = 0.05
val_prob_clipped = np.where(val_active_prob < PROB_FLOOR, 0, val_active_prob)
val_final = val_prob_clipped * val_consumption

# ── Tuning B: Recent-trend scaling ──
# The regressor was trained on all history (mean ~530 kWh) but recent consumption is lower.
# Compute a recency ratio from the last 30 active training days to calibrate.
recent_window = 30
recent_active = daily_train[daily_train['is_active'] == 1].tail(recent_window)
recent_mean = recent_active['y'].mean()
train_active_mean = train_active['y'].mean()
recency_ratio = recent_mean / train_active_mean if train_active_mean > 0 else 1.0
recency_ratio = np.clip(recency_ratio, 0.7, 1.3)  # guard against extreme shifts
val_final = val_final * recency_ratio

val_daily_preds = daily_val[['ds']].copy()
val_daily_preds['pred'] = val_final
val_daily_preds['clf_active'] = (val_active_prob >= 0.5).astype(int)
val_daily_preds['clf_prob'] = val_active_prob.round(3)

print(f"Tuning applied:")
print(f"  Soft blending: P(active) × regressor output  (floor={PROB_FLOOR})")
print(f"  Recency ratio: {recency_ratio:.3f}  (recent {recent_window}-day active mean: {recent_mean:.1f} vs training: {train_active_mean:.1f})")
print(f"\nTwo-stage validation predictions:")
print(f"  Hard-active (P≥0.5): {(val_active_prob >= 0.5).sum()}, soft-active (P≥{PROB_FLOOR}): {(val_active_prob >= PROB_FLOOR).sum()}")
print(f"  Actual active: {(daily_val['y'] >= SHUTDOWN_THRESH).sum()}, "
      f"shutdown: {(daily_val['y'] < SHUTDOWN_THRESH).sum()}")
val_daily_preds.head(10)

In [ ]:
# ── Step 5a: Merge daily actuals + predictions ──
daily_merged = daily_val[['ds', 'y']].copy()
daily_merged['pred'] = val_daily_preds['pred'].values
daily_merged['day_type'] = daily_merged['ds'].apply(classify_day)

# Classifier accuracy on validation
val_actual_active = (daily_val['y'] >= SHUTDOWN_THRESH).astype(int).values
val_pred_active = val_daily_preds['clf_active'].values
clf_correct = (val_actual_active == val_pred_active).sum()
print(f"═══ CLASSIFIER (Stage 1) ═══")
print(f"  Accuracy: {clf_correct}/{len(val_actual_active)} ({100*clf_correct/len(val_actual_active):.1f}%)")
tp = ((val_pred_active == 0) & (val_actual_active == 0)).sum()
fp = ((val_pred_active == 0) & (val_actual_active == 1)).sum()
fn = ((val_pred_active == 1) & (val_actual_active == 0)).sum()
tn = ((val_pred_active == 1) & (val_actual_active == 1)).sum()
print(f"  Shutdown detection: TP={tp}, FP={fp}, FN={fn}")
print(f"  (TP=correctly predicted shutdown, FP=active day wrongly called shutdown, FN=shutdown missed)")

In [ ]:
# ── Step 5b: Decompose daily → hourly using DOW profiles (primary) + day-type (fallback) ──
def decompose_daily_to_hourly(daily_preds_df, dow_prof, daytype_prof, dow_counts, min_days=4, col='pred'):
    """Convert daily predictions into 24 hourly values.
    Uses day-of-week profile if enough samples exist, else falls back to day-type profile."""
    rows = []
    for _, row in daily_preds_df.iterrows():
        day_ts = pd.Timestamp(row['ds'])
        day_total = max(row[col], 0)
        day_type = classify_day(day_ts)
        d = day_ts.dayofweek

        if d in dow_prof.columns and dow_counts.get(d, 0) >= min_days:
            profile = dow_prof[d].values
        else:
            profile = daytype_prof[day_type].values

        for h in range(24):
            rows.append({
                'ds': day_ts + pd.Timedelta(hours=h),
                'forecast_kwh': day_total * profile[h],
                'day_type': day_type,
            })
    return pd.DataFrame(rows)

val_hourly_preds = decompose_daily_to_hourly(
    daily_merged, dow_profiles, hourly_profiles, dow_day_counts.to_dict(),
    min_days=MIN_DAYS_FOR_DOW, col='pred'
)

val_start = daily_val['ds'].min()
val_end   = daily_val['ds'].max() + pd.Timedelta(hours=23)
actual_hourly_val = hourly_df[
    (hourly_df['rtc_timestamp'] >= val_start) &
    (hourly_df['rtc_timestamp'] <= val_end)
].copy()

val_merged = actual_hourly_val.merge(
    val_hourly_preds, left_on='rtc_timestamp', right_on='ds', how='inner')

# ── Metrics ──
# Daily (active days only: actual > 50 kWh/day)
active_days = daily_merged['y'] > 50
xgb_daily_mae  = mean_absolute_error(daily_merged.loc[active_days, 'y'], daily_merged.loc[active_days, 'pred'])
xgb_daily_rmse = root_mean_squared_error(daily_merged.loc[active_days, 'y'], daily_merged.loc[active_days, 'pred'])
xgb_daily_mape = np.mean(
    np.abs((daily_merged.loc[active_days, 'y'] - daily_merged.loc[active_days, 'pred'])
           / daily_merged.loc[active_days, 'y'])
) * 100

# Hourly (active hours: actual > 1 kWh)
xgb_hourly_mae  = mean_absolute_error(val_merged['consumption_kwh'], val_merged['forecast_kwh'])
xgb_hourly_rmse = root_mean_squared_error(val_merged['consumption_kwh'], val_merged['forecast_kwh'])
ACTIVE_HOUR_THRESH = 5.0
active_hrs = val_merged['consumption_kwh'] > ACTIVE_HOUR_THRESH
xgb_hourly_mape = np.mean(
    np.abs((val_merged.loc[active_hrs, 'consumption_kwh'] - val_merged.loc[active_hrs, 'forecast_kwh'])
           / val_merged.loc[active_hrs, 'consumption_kwh'])
) * 100

print(f"\n═══ DAILY-level metrics (active days only, y > 50 kWh) ═══")
print(f"  Active days: {active_days.sum()} / {len(daily_merged)}")
print(f"  MAE  : {xgb_daily_mae:.2f} kWh/day")
print(f"  RMSE : {xgb_daily_rmse:.2f} kWh/day")
print(f"  MAPE : {xgb_daily_mape:.2f}%")

print(f"\n═══ HOURLY-level metrics (after decomposition) ═══")
print(f"  MAE  : {xgb_hourly_mae:.2f} kWh/h")
print(f"  RMSE : {xgb_hourly_rmse:.2f} kWh/h")
print(f"  MAPE : {xgb_hourly_mape:.2f}%  (active hours, y > {ACTIVE_HOUR_THRESH} kWh, n={active_hrs.sum()})")

total_actual = daily_merged['y'].sum()
total_predicted = daily_merged['pred'].sum()
monthly_error_pct = (total_predicted - total_actual) / total_actual * 100

print(f"\n═══ MONTHLY TOTAL (validation period) ═══")
print(f"  Actual total     : {total_actual:,.0f} kWh")
print(f"  Predicted total  : {total_predicted:,.0f} kWh")
print(f"  Error            : {monthly_error_pct:+.2f}%")
print(f"  Abs difference   : {abs(total_predicted - total_actual):,.0f} kWh")

In [ ]:
# ── Step 5c: Hourly Correction Model ──
# The profile decomposition produces smooth curves; this model learns residual
# patterns (e.g. "hour 0 on Mondays is typically underestimated by the profile")

# Hourly correction model was tested but REMOVED — overfits with current data volume.
# Profile-only decomposition at ~41% hourly MAPE is the practical floor.
print('Correction model skipped. Profile decomposition is the final hourly approach.')


In [ ]:
# Validation plots
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# 1. Daily: actual vs scaled
bar_w = 0.35
x = np.arange(len(daily_merged))
axes[0].bar(x - bar_w, daily_merged['y'].values, width=bar_w, alpha=0.7,
            color='steelblue', label='Actual')
axes[0].bar(x, daily_merged['pred'].values, width=bar_w, alpha=0.8,
            color='green', label='Scaled prediction')
axes[0].set_title('Daily Totals — Actual vs Two-Stage XGBoost')
axes[0].set_ylabel('kWh / day')
axes[0].set_xticks(x[::2])
# axes[0].set_xticklabels([d.strftime('%d %b') for d in daily_merged['ds'].values[::2]], rotation=45)
axes[0].legend()

# 2. Hourly: actual vs decomposed
axes[1].plot(val_merged['rtc_timestamp'], val_merged['consumption_kwh'],
             label='Actual hourly', color='steelblue', linewidth=0.8, alpha=0.8)
axes[1].plot(val_merged['rtc_timestamp'], val_merged['forecast_kwh'],
             label='Decomposed hourly forecast', color='green', linewidth=0.9, linestyle='--')
axes[1].set_title('Hourly — Actual vs Decomposed (DOW Profile)')
axes[1].set_ylabel('kWh / hour')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

# 3. Zoomed last 7 days
zoom_start = val_merged['rtc_timestamp'].max() - pd.Timedelta(days=7)
zoom = val_merged[val_merged['rtc_timestamp'] >= zoom_start]
axes[2].plot(zoom['rtc_timestamp'], zoom['consumption_kwh'],
             label='Actual', color='steelblue', linewidth=1.2)
axes[2].plot(zoom['rtc_timestamp'], zoom['forecast_kwh'],
             label='Decomposed (DOW profile)', color='green', linewidth=1.1, linestyle='--')
axes[2].set_title('Zoomed — Last 7 Days of Validation')
axes[2].set_ylabel('kWh / hour')
axes[2].legend()
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))

for ax in axes:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()

## 11. Forecast Future Month — Two-Stage XGBoost → Hourly Decomposition

In [ ]:
# Retrain two-stage model on ALL daily data, then forecast future month

# Stage 1: classifier on all data
xgb_clf_full = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(daily_df['is_active'] == 0).sum() / max((daily_df['is_active'] == 1).sum(), 1),
    random_state=42, eval_metric='logloss',
)
xgb_clf_full.fit(daily_df[DAILY_FEATURES], daily_df['is_active'], verbose=False)

# Stage 2: regressor on active days only
active_full = daily_df[daily_df['is_active'] == 1]
xgb_reg_full = XGBDaily(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42,
)
xgb_reg_full.fit(active_full[DAILY_FEATURES], active_full['y'], verbose=False)

# ── Recency ratio for full model ──
recent_full_active = daily_df[daily_df['is_active'] == 1].tail(recent_window)
recency_ratio_full = recent_full_active['y'].mean() / active_full['y'].mean()
recency_ratio_full = np.clip(recency_ratio_full, 0.7, 1.3)

last_day = daily_df['ds'].max()
future_days = pd.date_range(last_day + pd.Timedelta(days=1), periods=DAILY_HORIZON, freq='D')

# Build deterministic features for future
futr = pd.DataFrame({'ds': future_days})
futr['is_holiday']    = futr['ds'].dt.date.apply(lambda d: 1 if d in _gj_hols_nf else 0)
futr['is_weekend']    = (futr['ds'].dt.dayofweek >= 5).astype(int)
futr['is_working_day'] = ((futr['is_weekend'] == 0) & (futr['is_holiday'] == 0)).astype(int)
futr['dow']           = futr['ds'].dt.dayofweek
futr['dow_sin']       = np.sin(2 * np.pi * futr['dow'] / 7)
futr['dow_cos']       = np.cos(2 * np.pi * futr['dow'] / 7)
futr['month']         = futr['ds'].dt.month
futr['mon_sin']       = np.sin(2 * np.pi * futr['month'] / 12)
futr['mon_cos']       = np.cos(2 * np.pi * futr['month'] / 12)
futr['day_of_month']  = futr['ds'].dt.day

# ── Tuned two-stage prediction (same logic as validation) ──
futr_active_prob = xgb_clf_full.predict_proba(futr[DAILY_FEATURES])[:, 1]
futr_consumption = np.clip(xgb_reg_full.predict(futr[DAILY_FEATURES]), 0, None)
futr_prob_clipped = np.where(futr_active_prob < PROB_FLOOR, 0, futr_active_prob)
future_preds = futr_prob_clipped * futr_consumption * recency_ratio_full

future_daily = pd.DataFrame({'ds': future_days, 'daily_kwh': future_preds})
future_daily['day_type'] = future_daily['ds'].apply(classify_day)
future_daily['predicted_active'] = (futr_active_prob >= 0.5).astype(int)

# Decompose into hourly using DOW profiles (primary) + day-type (fallback)
future_hourly = decompose_daily_to_hourly(
    future_daily, dow_profiles, hourly_profiles, dow_day_counts.to_dict(),
    min_days=MIN_DAYS_FOR_DOW, col='daily_kwh'
)

# Hourly correction model removed — overfits with current data volume

print(f"Future daily : {future_daily['ds'].min().date()} → {future_daily['ds'].max().date()}")
print(f"Future hourly: {future_hourly['ds'].min()} → {future_hourly['ds'].max()}")
print(f"Total forecasted: {future_daily['daily_kwh'].sum():,.2f} kWh")
print(f"Recency ratio (full): {recency_ratio_full:.3f}")
print(f"Hourly rows: {len(future_hourly)}")
print(f"\nDay type breakdown:")
print(future_daily.groupby('day_type')['daily_kwh'].agg(['count', 'mean', 'sum']).round(1))

In [ ]:
# History + future hourly forecast
last_14d = hourly_df[hourly_df['rtc_timestamp'] >= hourly_df['rtc_timestamp'].max() - pd.Timedelta(days=14)]

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(last_14d['rtc_timestamp'], last_14d['consumption_kwh'],
        label='Historical (last 14 days)', color='steelblue', linewidth=0.7, alpha=0.8)
ax.plot(future_hourly['ds'], future_hourly['forecast_kwh'],
        label='Future Month — Hourly Forecast', color='green', linewidth=0.8)
ax.axvline(hourly_df['rtc_timestamp'].max(), color='black', linewidth=1.5, linestyle=':', label='Forecast start')
ax.set_title('Hourly Forecast — Historical Tail + Future Month (XGBoost Daily × Profiles)')
ax.set_ylabel('kWh / hour')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Daily and weekly summary with day-type coloring
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Color bars by day type
color_map = {'working': '#2ecc71', 'weekend': '#e74c3c', 'holiday': '#f39c12'}
bar_colors = [color_map[dt] for dt in future_hourly.groupby(future_hourly['ds'].dt.date)['day_type'].first().values]

axes[0].bar(future_daily['ds'], future_daily['daily_kwh'], color=bar_colors, alpha=0.8, width=0.8)
axes[0].set_title('Forecasted Daily Energy Consumption — Next Month')
axes[0].set_ylabel('Total kWh / day')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Working'),
                   Patch(facecolor='#e74c3c', label='Weekend'),
                   Patch(facecolor='#f39c12', label='Holiday')]
axes[0].legend(handles=legend_elements)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Weekly
weekly_forecast = (
    future_hourly.set_index('ds')['forecast_kwh']
    .resample('W').sum()
    .reset_index()
)
axes[1].bar(weekly_forecast['ds'], weekly_forecast['forecast_kwh'], color='teal', alpha=0.7, width=5)
axes[1].set_title('Forecasted Weekly Energy Consumption')
axes[1].set_ylabel('Total kWh / week')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

total_kwh = future_daily['daily_kwh'].sum()
avg_daily = future_daily['daily_kwh'].mean()
print(f"Total forecasted energy for next month : {total_kwh:,.2f} kWh")
print(f"Average daily consumption               : {avg_daily:,.2f} kWh/day")
print(f"Peak day   : {future_daily.loc[future_daily['daily_kwh'].idxmax(), 'ds'].strftime('%d %b %Y')} ({future_daily['daily_kwh'].max():,.2f} kWh)")
print(f"Lowest day : {future_daily.loc[future_daily['daily_kwh'].idxmin(), 'ds'].strftime('%d %b %Y')} ({future_daily['daily_kwh'].min():,.2f} kWh)")

In [ ]:
# Compare model summaries
summary = pd.DataFrame([
    {'Model': 'XGBoost (1-step hourly)',
     'Granularity': 'Hourly', 'MAE': xgb_mae, 'RMSE': xgb_rmse,
     'MAPE%': xgb_mape, 'Use case': 'Next 1h to 1-2 days'},
    {'Model': 'Two-stage XGBoost + profiles',
     'Granularity': 'Daily', 'MAE': xgb_daily_mae, 'RMSE': xgb_daily_rmse,
     'MAPE%': xgb_daily_mape, 'Use case': 'Next month (daily totals)'},
    {'Model': 'Two-stage XGBoost + DOW profiles',
     'Granularity': 'Hourly (decomposed)', 'MAE': xgb_hourly_mae, 'RMSE': xgb_hourly_rmse,
     'MAPE%': xgb_hourly_mape, 'Use case': 'Next month (hourly detail)'},
])
summary.round(4)

In [ ]:
# Save forecast + profiles + model
import joblib

future_hourly.to_csv(f"{MODEL_DIR}/future_hourly_forecast.csv", index=False)
future_daily.to_csv(f"{MODEL_DIR}/future_daily_forecast.csv", index=False)
hourly_profiles.to_csv(f"{MODEL_DIR}/hourly_profiles_by_daytype.csv")
dow_profiles.to_csv(f"{MODEL_DIR}/hourly_profiles_by_dow.csv")
joblib.dump(xgb_clf_full, f"{MODEL_DIR}/xgb_daily_classifier.pkl")
joblib.dump(xgb_reg_full, f"{MODEL_DIR}/xgb_daily_regressor.pkl")
print('Models saved: classifier, regressor, DOW + day-type hourly profiles, and forecasts.')

## 12. Hybrid 48h Forecast — XGBoost + Two-Stage Daily × DOW Profile

> **Hours 1-6**: XGBoost recursive (real lag data, best short-term accuracy)  
> **Hours 7-48**: Two-stage daily model + DOW profile decomposition  
> Uses hard P(active) threshold (15%) instead of soft blending to preserve realistic daily totals for weekends/holidays.

In [ ]:
def forecast_recursive(model, history_df: pd.DataFrame, steps: int = 24) -> pd.DataFrame:
    """
    Recursive 1-step-at-a-time XGBoost forecasting (no weather needed).

    Parameters
    ----------
    model       : fitted pipeline that predicts consumption_kwh.
    history_df  : continuous hourly DataFrame with at least
                  ['rtc_timestamp', 'total_kwh', 'consumption_kwh'].
    steps       : number of 1-hour steps to forecast.
    """
    base_cols = ['rtc_timestamp', 'total_kwh', 'consumption_kwh']
    history = history_df[base_cols].copy().reset_index(drop=True)

    last_ts = history['rtc_timestamp'].iloc[-1]
    future_ts = pd.date_range(last_ts + pd.Timedelta(hours=1), periods=steps, freq='1h')

    _gj_hols = holidays.India(state='GJ', years=range(2023, 2027))
    holiday_dates = set(_gj_hols.keys())
    preds = []

    for next_ts in future_ts:
        row = {}

        # Cyclical time
        row['hour_sin'] = np.sin(2 * np.pi * next_ts.hour / 24)
        row['hour_cos'] = np.cos(2 * np.pi * next_ts.hour / 24)
        row['dow_sin']  = np.sin(2 * np.pi * next_ts.dayofweek / 7)
        row['dow_cos']  = np.cos(2 * np.pi * next_ts.dayofweek / 7)
        row['mon_sin']  = np.sin(2 * np.pi * next_ts.month / 12)
        row['mon_cos']  = np.cos(2 * np.pi * next_ts.month / 12)

        # kWh lag features
        for lag in [1, 2, 3, 6, 12, 24]:
            row[f'kwh_lag_{lag}'] = (
                history['total_kwh'].iloc[-lag] if lag <= len(history) else np.nan
            )

        # Rolling stats on total_kwh
        last6 = history['total_kwh'].iloc[-6:]
        row['rolling_max_6']   = last6.max()
        row['rolling_min_6']   = last6.min()
        row['rolling_range_6'] = row['rolling_max_6'] - row['rolling_min_6']
        for w in [3, 6, 24]:
            tail = history['total_kwh'].iloc[-w:]
            row[f'kwh_roll_std_{w}'] = tail.std() if len(tail) >= 2 else 0.0
        row['diff_2'] = history['total_kwh'].iloc[-1] - history['total_kwh'].iloc[-3] if len(history) >= 3 else 0.0

        # Consumption lags
        for lag in [1, 2, 3, 6, 24]:
            row[f'cons_lag_{lag}'] = (
                history['consumption_kwh'].iloc[-lag] if lag <= len(history) else np.nan
            )
        row['cons_roll_mean_6']  = history['consumption_kwh'].iloc[-6:].mean()
        row['cons_roll_mean_24'] = history['consumption_kwh'].iloc[-24:].mean()

        # Working calendar
        row['is_holiday'] = int(next_ts.date() in _gj_hols)
        row['is_working_day'] = int(next_ts.dayofweek < 5 and row['is_holiday'] == 0)
        row['is_working_hour'] = int(
            9 <= next_ts.hour <= 18 and row['is_working_day'] == 1
        )
        _dnth = 7
        for _i in range(1, 8):
            if (next_ts + pd.Timedelta(days=_i)).date() in holiday_dates:
                _dnth = _i; break
        row['days_to_next_holiday'] = _dnth

        # Shift
        _h = next_ts.hour
        _shift = 1 if 6 <= _h < 14 else (2 if 14 <= _h < 22 else 0)
        row['shift']     = _shift
        row['shift_sin'] = np.sin(2 * np.pi * _shift / 3)
        row['shift_cos'] = np.cos(2 * np.pi * _shift / 3)

        # Time since last shutdown
        _recent = history['consumption_kwh'].iloc[-48:]
        _zeros = (_recent < 0.5)
        if _zeros.all():
            row['hours_since_shutdown'] = 0
            row['just_restarted']       = 0
        elif not _zeros.any():
            row['hours_since_shutdown'] = min(len(_recent), 168)
            row['just_restarted']       = 0
        else:
            _last_zero_idx = _zeros[_zeros].index[-1]
            _hss = int(len(_recent) - _recent.index.get_loc(_last_zero_idx) - 1)
            row['hours_since_shutdown'] = min(_hss, 168)
            row['just_restarted']       = int(0 < _hss <= 2)

        # Predict consumption
        X_row = pd.DataFrame([row]).reindex(columns=model_features)
        pred_cons = max(float(model.predict(X_row)[0]), 0.0)

        # Anti-convergence: blend with same-hour-yesterday to preserve daily cycle.
        # Without this, recursive lags converge to a constant after ~6 steps.
        step_num = len(preds)
        if len(history) >= 24:
            yesterday_cons = max(float(history['consumption_kwh'].iloc[-24]), 0.0)
            model_weight = max(0.35, 1.0 - step_num * 0.025)
            pred_cons = model_weight * pred_cons + (1 - model_weight) * yesterday_cons

        last_kwh = float(history['total_kwh'].iloc[-1])
        pred_total = last_kwh + pred_cons

        preds.append({
            'rtc_timestamp': next_ts,
            'consumption_kwh': pred_cons,
            'total_kwh': pred_total,
        })

        new_row = {'rtc_timestamp': next_ts, 'total_kwh': pred_total, 'consumption_kwh': pred_cons}
        history = pd.concat([history, pd.DataFrame([new_row])], ignore_index=True)

    return pd.DataFrame(preds)


# Hybrid 48h forecast 
# Hours 1-6:  XGBoost 1-step recursive (real lag data, best short-term accuracy)
# Hours 7-48: Two-stage daily model + DOW profile decomposition
#             (uses regressor directly for likely-active days, only zeroes very-low-prob days)
HANDOFF_HOUR = 6

# Part 1: XGBoost recursive for first 6 hours
short_preds = forecast_recursive(xgb_model, df, steps=HANDOFF_HOUR)

# Part 2: Two-stage model → DOW profiles for hours 7-48
last_ts = df['rtc_timestamp'].iloc[-1]
future_48h_start = last_ts + pd.Timedelta(hours=1)
future_48h_ts = pd.date_range(future_48h_start, periods=52, freq='1h')

future_dates = pd.Series(future_48h_ts.date).unique()
futr_48 = pd.DataFrame({'ds': pd.to_datetime(future_dates)})
futr_48['is_holiday'] = futr_48['ds'].dt.date.apply(lambda d: 1 if d in _gj_hols_nf else 0)
futr_48['is_weekend']  = (futr_48['ds'].dt.dayofweek >= 5).astype(int)
futr_48['is_working_day'] = ((futr_48['is_weekend'] == 0) & (futr_48['is_holiday'] == 0)).astype(int)
futr_48['dow'] = futr_48['ds'].dt.dayofweek
futr_48['dow_sin'] = np.sin(2 * np.pi * futr_48['dow'] / 7)
futr_48['dow_cos'] = np.cos(2 * np.pi * futr_48['dow'] / 7)
futr_48['month'] = futr_48['ds'].dt.month
futr_48['mon_sin'] = np.sin(2 * np.pi * futr_48['month'] / 12)
futr_48['mon_cos'] = np.cos(2 * np.pi * futr_48['month'] / 12)
futr_48['day_of_month'] = futr_48['ds'].dt.day

# For 48h: use HARD threshold (not soft blend) — only zero out high-confidence shutdown days.
# Soft blending over-penalizes weekends/holidays for short-term where we want realistic levels.
SHORT_TERM_SHUTDOWN_THRESH = 0.15  # only zero out if P(active) < 15%
prob_48 = xgb_clf_full.predict_proba(futr_48[DAILY_FEATURES])[:, 1]
cons_48 = np.clip(xgb_reg_full.predict(futr_48[DAILY_FEATURES]), 0, None)

# Use regressor output directly, only zero out very-confident shutdown days
daily_preds_48 = np.where(prob_48 >= SHORT_TERM_SHUTDOWN_THRESH, cons_48, 0)

# Still apply recency scaling to bring regressor closer to recent levels
daily_preds_48 = daily_preds_48 * recency_ratio_full

daily_48 = pd.DataFrame({'ds': pd.to_datetime(future_dates), 'daily_kwh': daily_preds_48})
daily_48['day_type'] = daily_48['ds'].apply(classify_day)
daily_48['p_active'] = prob_48

# Decompose into hourly using DOW profiles
hourly_48 = decompose_daily_to_hourly(
    daily_48, dow_profiles, hourly_profiles, dow_day_counts.to_dict(),
    min_days=MIN_DAYS_FOR_DOW, col='daily_kwh'
)
hourly_48 = hourly_48[hourly_48['ds'].isin(future_48h_ts)].reset_index(drop=True)

# Stitch: XGBoost for hours 1-6, smooth handoff, daily model for 7-48 
next_48h_rows = []
for i, ts in enumerate(future_48h_ts):
    hour_num = i + 1
    xgb_row = short_preds.loc[short_preds['rtc_timestamp'] == ts]
    prof_row = hourly_48.loc[hourly_48['ds'] == ts]

    xgb_kwh = float(xgb_row['consumption_kwh'].iloc[0]) if len(xgb_row) > 0 else None
    prof_kwh = float(prof_row['forecast_kwh'].iloc[0]) if len(prof_row) > 0 else 0.0

    if hour_num <= 3 and xgb_kwh is not None:
        final = xgb_kwh
    elif hour_num <= HANDOFF_HOUR and xgb_kwh is not None:
        w = (hour_num - 3) / (HANDOFF_HOUR - 3)
        final = (1 - w) * xgb_kwh + w * prof_kwh
    else:
        final = prof_kwh

    next_48h_rows.append({'rtc_timestamp': ts, 'consumption_kwh': max(final, 0)})

next_48h = pd.DataFrame(next_48h_rows)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
last_7d = df.tail(24 * 7)
ax.plot(last_7d['rtc_timestamp'], last_7d['consumption_kwh'],
        label='Historical (last 7 days)', color='steelblue', linewidth=1.0)
ax.plot(next_48h['rtc_timestamp'], next_48h['consumption_kwh'],
        label='Forecast (next 48h)', color='red', linewidth=1.3, linestyle='--')
ax.axvline(df['rtc_timestamp'].iloc[-1], color='black', linestyle=':', linewidth=1.5, label='Now')
ax.set_title('Hybrid 48h Forecast — XGBoost (1-6h) + Two-Stage Daily × DOW Profile (7-48h)')
ax.set_ylabel('kWh / hour')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nForecasted total for next 48h: {next_48h['consumption_kwh'].sum():.2f} kWh")
print(f"Forecasted avg hourly:  {next_48h['consumption_kwh'].mean():.2f} kWh/h")
print(f"Recency ratio:  {recency_ratio_full:.3f}")
print(f"\nDaily breakdown:")
for _, row in daily_48.iterrows():
    d = row['ds'].date()
    dt = row['day_type']
    kwh = row['daily_kwh']
    pa = row['p_active']
    print(f"  {d} ({dt}): {kwh:.1f} kWh  [P(active)={pa:.2f}, regressor={float(cons_48[daily_48.index[daily_48['ds'] == row['ds']][0]]):.1f}]")